In [1]:
import pandas as pd

In [2]:
customer_side_features = pd.read_csv("../data/processed/customer_side_features.csv")
customer_summary = pd.read_csv("../data/processed/customer_summary.csv")
monthly_sales_trend = pd.read_csv("../data/processed/monthly_sales_trend.csv")
moving_averages_summary = pd.read_csv("../data/processed/moving_averages_summary.csv")
price_and_promotion = pd.read_csv("../data/processed/price_and_promotion.csv")
product_consistency_performance = pd.read_csv("../data/processed/product_consistency_performance.csv")
product_cust_behaviour = pd.read_csv("../data/processed/product_customer_behaviour.csv")
product_summary = pd.read_csv("../data/processed/product_summary.csv")
sales_transaction_summary = pd.read_csv("../data/processed/sales_transaction_summary.csv")
store_monthly_summary = pd.read_csv("../data/processed/store_monthly_summary.csv")
store_side_features = pd.read_csv("../data/processed/store_side_features.csv")
store_summary = pd.read_csv("../data/processed/store_summary.csv")
store_yearly_summary = pd.read_csv("../data/processed/store_yearly_summary.csv")
time_lag_features = pd.read_csv("../data/processed/time_lag_features.csv")
product_customer_behaviour_grain_level = pd.read_csv("../data/processed/product_customer_behaviour_grain_level.csv")
profit_and_market_share = pd.read_csv("../data/processed/profit_and_market_share.csv")

In [4]:
dataframes = {
    "customer_side_features": customer_side_features,
    "customer_summary": customer_summary,
    "monthly_sales_trend": monthly_sales_trend,
    "moving_averages_summary": moving_averages_summary,
    "price_and_promotion": price_and_promotion,
    "product_customer_behaviour" : product_cust_behaviour,
    "product_consistency_performance": product_consistency_performance,
    "product_summary": product_summary,
    "sales_transaction_summary" : sales_transaction_summary,
    "store_monthly_summary" : store_monthly_summary,
    "store_side_features": store_side_features,
    "store_summary": store_summary,
    "store_yearly_summary": store_yearly_summary,
    "time_lag_features": time_lag_features,
    "product_customer_behaviour_grain_level": product_customer_behaviour_grain_level,
    "profit_and_market_share": profit_and_market_share
}

In [5]:
def feature_audit(source_table):
    audit_results = []
    for source_table, df in dataframes.items():
        for col in df.select_dtypes(include='number').columns:
            source_table = source_table
            feature_name = col
            data_type = df[col].dtypes
            null_pct = (df[col].isnull().sum() / df[col].shape[0])*100
            n_unique = df[col].nunique()
            duplicated_pct = (df[col].duplicated().sum() / df[col].shape[0])*100
            constant_or_zero_variance = df[col].nunique(dropna=True) <= 1
            min_val = df[col].describe().loc['min']
            max_val = df[col].describe().loc['max']
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5*IQR
            upper = Q3 + 1.5*IQR
            outliers = ((df[col] < lower) | (df[col] > upper)).sum()
            outlier_pct = (outliers/df[col].notna().sum())*100
            feature = pd.DataFrame({"source_table": [source_table], "feature_name": [feature_name], "data_type": [data_type], "null_pct": [null_pct], 
                                    "n_unique": [n_unique], "duplicated_pct": [duplicated_pct], "constant_or_zero_variance": [constant_or_zero_variance],
                                    "min": [min_val], "max": [max_val], "outlier_pct": [outlier_pct]})
            audit_results.append(feature)

    audit_df = pd.concat(audit_results, ignore_index=True)

    return audit_df

In [6]:
dataframes.keys()

dict_keys(['customer_side_features', 'customer_summary', 'monthly_sales_trend', 'moving_averages_summary', 'price_and_promotion', 'product_customer_behaviour', 'product_consistency_performance', 'product_summary', 'sales_transaction_summary', 'store_monthly_summary', 'store_side_features', 'store_summary', 'store_yearly_summary', 'time_lag_features', 'product_customer_behaviour_grain_level', 'profit_and_market_share'])

In [7]:
audit_final_data = feature_audit(dataframes)

In [8]:
audit_final_data.shape

(126, 10)

In [11]:
audit_final_data.to_excel("../src/preprocessing/Master_Feature_Audit_1.xlsx", index=False)

In [12]:
audit_final_data.head()

,source_table,feature_name,data_type,null_pct,n_unique,duplicated_pct,constant_or_zero_variance,min,max,outlier_pct
0,customer_side_features,purchase_frequency,float64,0.0,77,99.23,False,0.35,1.21,0.48
1,customer_side_features,purchase_consistency,float64,0.0,61,99.39,False,0.00,0.90,3.70
2,customer_side_features,repeat_purchase_rate,float64,0.0,47,99.53,False,94.74,98.57,2.95
3,customer_side_features,average_discount_received,float64,0.0,539,94.61,False,7.13,14.76,0.85
4,customer_side_features,average_price_paid,float64,0.0,9929,0.71,False,2874.68,23874.50,1.41
